# Project 14 — State switching over time (Hidden Markov Model)

**Scenario.** An ion channel (or an MD trajectory) switches between a **closed** and an **open** state over time. The state *persists* — once open it tends to stay open — so successive noisy emissions are correlated through the hidden state. We observe the emissions, never the states, and want the **transition matrix** and the per-state emission means.

**New skill.** Discrete latent *dynamics* and transition-matrix inference. **Key pitfall.** Non-identifiable state labels (same permutation symmetry as a mixture). **Implementation.** We **marginalize the discrete states** with the forward algorithm (in `pytensor.scan`, inside a `pm.Potential`), so NUTS only samples continuous parameters.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

Generative model: $s_1\sim\text{Cat}(\pi)$, $s_t\mid s_{t-1}\sim\text{Cat}(P[s_{t-1},:])$, $y_t\mid s_t\sim\mathcal{N}(\mu_{s_t},\sigma)$. The transition matrix is parameterized by two switch probabilities $p_{01}$ (closed→open) and $p_{10}$ (open→closed). **Assumptions:** (a) exactly 2 states, (b) first-order Markov (memoryless given the previous state), (c) Gaussian emissions with shared $\sigma$, (d) time-homogeneous transitions. Truth: $p_{01}=0.08, p_{10}=0.15, \mu=(-1.5,1.5), \sigma=0.6$.

In [ ]:
from data.generate_data import generate
data = generate(); y = data['y']; states = data['states']; t = data['truth']
print(f"T={len(y)}, fraction open={np.mean(states==1):.2f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8,4), sharex=True)
axes[0].plot(y, lw=0.8, color='#4C72B0'); axes[0].set_ylabel('emission y_t')
axes[1].step(range(len(states)), states, where='mid', color='#C44E52')
axes[1].set(ylabel='hidden state', yticks=[0,1], xlabel='time t')
axes[0].set_title('Noisy emissions and the (hidden) state path')
plt.tight_layout()

## Step 2 — Model: marginalize the states (forward algorithm)

We do **not** sample the discrete path $s_{1:T}$. The marginal likelihood $p(y\mid\theta)=\sum_{s_{1:T}} p(y,s_{1:T}\mid\theta)$ is computed exactly by the forward recursion in log space:

$$\alpha_1[j]=\log\pi_j+\log\mathcal N(y_1;\mu_j,\sigma),\quad \alpha_t[j]=\log\mathcal N(y_t;\mu_j,\sigma)+\operatorname*{logsumexp}_i\big(\alpha_{t-1}[i]+\log P_{ij}\big),$$

$$\log p(y\mid\theta)=\operatorname*{logsumexp}_j \alpha_T[j].$$

The **log-sum-exp at every step is essential** — it is what makes this the *sum* over paths (a proper marginal) rather than the *max* over paths (Viterbi). Priors: $p_{01},p_{10}\sim\text{Beta}(2,8)$ (favouring rare switches), $\mu\sim\text{Normal}(0,3)$ **ordered** so $\mu_0<\mu_1$ (breaks the state-label symmetry), $\sigma\sim\text{HalfNormal}(1)$.

In [ ]:
from model import build_model, fit, add_named
model = build_model(data, ordered=True)
model

## Step 3 — Prior predictive (on parameters)

Because the likelihood is a `pm.Potential` (no observed RV), standard data-space prior predictive is not defined. We instead inspect the *parameter* prior: do the implied dwell times $1/p$ and emission means look physically plausible? Beta(2,8) puts switch probabilities mostly in 0.05–0.4, i.e. dwell times of a few to tens of steps — reasonable for a channel.

In [ ]:
from scipy.stats import beta as beta_dist
grid = np.linspace(0,1,200)
fig, ax = plt.subplots(figsize=(6,3.2))
ax.plot(grid, beta_dist.pdf(grid, 2, 8), color='#55A868')
ax.fill_between(grid, beta_dist.pdf(grid, 2, 8), alpha=0.3, color='#55A868')
ax.set(xlabel='switch probability', ylabel='prior density',
       title='Beta(2,8) prior on p01, p10 — favours rare switches')
plt.tight_layout()

## Step 4 — Inference (NUTS)

Settings: `draws=500, tune=1000, chains=2, target_accept=0.9`. The forward scan makes each gradient ~T times more expensive than a plain model, so the fit takes ~1 minute at T=250 — still light. The ordered transform keeps the state labels identified.

In [ ]:
idata = fit(data, draws=500, tune=1000, chains=2, seed=14)
add_named(idata)

## Step 5 — Diagnostics & recovering the transition matrix

Check R-hat ≈ 1.00, healthy ESS, 0 divergences. Then read off the recovered transition probabilities and emission means and compare to truth.

In [ ]:
print(az.summary(idata, var_names=['p01','p10','mu','sigma','separation']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))

In [ ]:
az.plot_trace(idata, var_names=['p01','p10','mu']); plt.tight_layout()

In [ ]:
p01 = float(idata.posterior['p01'].mean()); p10 = float(idata.posterior['p10'].mean())
P_hat = np.array([[1-p01, p01],[p10, 1-p10]])
print('Recovered transition matrix P:\n', np.round(P_hat,3))
print('True P:\n', np.array([[1-t['p01'], t['p01']],[t['p10'], 1-t['p10']]]))

## Step 6 — Posterior predictive check (simulate trajectories)

We simulate trajectories from the posterior parameters and compare summary statistics (here the marginal emission histogram and the fraction of time in the high state) to the observed data — a posterior-predictive check tailored to a Potential-likelihood model.

In [ ]:
def sim_traj(p01, p10, mu, sigma, T, rng):
    P = np.array([[1-p01,p01],[p10,1-p10]]); s = np.zeros(T, int)
    s[0] = rng.integers(2)
    for k in range(1,T): s[k] = rng.choice(2, p=P[s[k-1]])
    return rng.normal(np.array(mu)[s], sigma)
rng = np.random.default_rng(0)
post = idata.posterior
draws = [(float(post['p01'].values.ravel()[i]), float(post['p10'].values.ravel()[i]),
          post['mu'].values.reshape(-1,2)[i], float(post['sigma'].values.ravel()[i]))
         for i in rng.integers(0, post['p01'].size, 40)]
fig, ax = plt.subplots(figsize=(6,3.5))
for d in draws:
    ax.hist(sim_traj(*d, len(y), rng), bins=30, histtype='step', density=True, alpha=0.3, color='#4C72B0')
ax.hist(y, bins=30, density=True, color='k', histtype='step', lw=2, label='observed')
ax.legend(); ax.set_title('Posterior predictive emission histograms'); plt.tight_layout()

## Step 7 — Criticism & identifiability

The state labels are identified only because we ordered the emission means. Report label-invariant quantities (separation, sigma) and *direction-aware* transition probabilities (p01 = into the higher state). `test_recovery.py` checks all of these against truth.

## Step 8 — Decision & communication

For a collaborator: 'The channel spends ~37% of the time open; it opens rarely (p≈0.08 per step) but, once open, closes at p≈0.15 per step, giving a mean open dwell of ~1/0.15 ≈ 7 steps.' Dwell times and occupancy are the actionable outputs. See `summary_onepager.md`.

In [ ]:
open_dwell = 1/float(idata.posterior['p10'].mean())
closed_dwell = 1/float(idata.posterior['p01'].mean())
print(f'mean open dwell  ~ {open_dwell:.1f} steps')
print(f'mean closed dwell~ {closed_dwell:.1f} steps')